In [ ]:
import polars as pl
import os
import sys
sys.path.append(os.path.abspath('..'))
import utilitaries.timestamp_sampling_utils as tsu
import utilitaries.extract_data_utils as extract
import utilitaries.create_merged_dataset as create_merged_dataset
DATASET_PATH = "/../../../data2/paquie.d/Datasets/output"
dataset_path = DATASET_PATH

In [ ]:
_path = os.path.join(dataset_path, 'df_static_full_clean.parquet')
df_static_no_ano = pl.read_parquet(_path)
print(df_static_no_ano)
print(df_static_no_ano.select("lifeTimeNumber").unique())
print(df_static_no_ano.columns)

In [ ]:
df_list = df_static_no_ano.group_by("lifeTimeNumber").agg(
    pl.col("encounterId").implode().alias("encounters")
)

print(df_list.filter(pl.col("encounters").list.len() >= 2))

In [ ]:
import polars as pl

# En partant de ton DataFrame actuel (nommé par exemple df_list)
df_avec_count = df_list.filter(pl.col("encounters").list.len() >= 2).select(
    pl.col("encounters").list.len()).sum().item()

print(df_avec_count)


In [ ]:
df_static_no_ano.filter(pl.col("isDeceased") == 1)

In [ ]:
df_static = pl.read_parquet(os.path.join(dataset_path, 'df_static_ano_clean.parquet'))
res = tsu.prepare_labels(df_static, 'absolute')

In [ ]:
print("Nombre de patient dc_h24", res.filter(pl.col("isDeceased_lt_24h") == 1).height)
print("Nombre de patient dc_j7", res.filter(pl.col("isDeceased_lt_7d") == 1).height)
print("Nombre de patient dc_j28", res.filter(pl.col("isDeceased_lt_28d") == 1).height)
print("Nombre de patient dc_3m", res.filter(pl.col("isDeceased_lt_3m") == 1).height)

In [ ]:
df_static = pl.read_parquet(os.path.join(dataset_path, 'df_static_ano_clean.parquet'))
df_keep_eId = (
    df_static_no_ano.sort(["isDeceased", "utcInTime"], descending=[True, True])
      .unique(subset=["lifeTimeNumber"], keep="first").filter(pl.col("isDeceased") == 1)
).select("encounterId")
df_static = df_static.join(df_keep_eId, on = "encounterId", how = "inner")
res = tsu.prepare_labels(df_static, 'absolute')

In [ ]:
print("Nombre de patient dc_h24", res.filter(pl.col("isDeceased_lt_24h") == 1).height)
print("Nombre de patient dc_j7", res.filter(pl.col("isDeceased_lt_7d") == 1).height)
print("Nombre de patient dc_j28", res.filter(pl.col("isDeceased_lt_28d") == 1).height)
print("Nombre de patient dc_3m", res.filter(pl.col("isDeceased_lt_3m") == 1).height)

In [ ]:
df_static = pl.read_parquet(os.path.join(dataset_path, 'df_static_ano_clean.parquet'))
df_keep_eId = (
    df_static_no_ano.sort(["utcInTime"], descending=[True])
      .unique(subset=["lifeTimeNumber"], keep="first").filter(pl.col("isDeceased") == 1)
).select("encounterId")
df_static = df_static.join(df_keep_eId, on = "encounterId", how = "inner")
res = tsu.prepare_labels(df_static, 'absolute')

In [ ]:
print("Nombre de patient dc_h24", res.filter(pl.col("isDeceased_lt_24h") == 1).height)
print("Nombre de patient dc_j7", res.filter(pl.col("isDeceased_lt_7d") == 1).height)
print("Nombre de patient dc_j28", res.filter(pl.col("isDeceased_lt_28d") == 1).height)
print("Nombre de patient dc_3m", res.filter(pl.col("isDeceased_lt_3m") == 1).height)

In [ ]:
_path = os.path.join(dataset_path, 'df_dynamic_full_clean.parquet')
df_dynamic = extract.extract_data_survie(_path)
df_static = pl.read_parquet(os.path.join(dataset_path, 'df_static_ano_clean.parquet'))
df_merged = create_merged_dataset.create_merged_dataset(df_static, df_dynamic, True, save=True, folder=dataset_path)
res = tsu.prepare_labels(df_merged, 'relative')

In [ ]:
df_keep_eId = (
    df_static_no_ano.sort(["isDeceased", "utcInTime"], descending=[True, True])
      .unique(subset=["lifeTimeNumber"], keep="first").filter(pl.col("isDeceased") == 1)
).select("encounterId")

In [ ]:
df_keep_eId

In [ ]:
_path = os.path.join(dataset_path, 'df_static_ano_clean.parquet')
df_static = pl.read_parquet(_path)
with pl.Config(tbl_rows=-1, tbl_width_chars=200):
    display(df_static["icu_mode_entree"].value_counts().sort("count", descending=True))